In [9]:
import pandas as pd

df = pd.read_csv('../resources/train.csv')
print(f"Jumlah data mentah awal: {len(df)}")

Jumlah data mentah awal: 55643


In [10]:
df.head()

[Output tidak disertakan pada rilis publik. Hanya statistik agregat yang dipublikasikan; teks pesan pengguna, daftar frekuensi kata, dan identitas pengguna dihilangkan. Lihat 'Data & Content Policy' di README.]


In [11]:
train_base = df.drop(columns=["Label Toxic", "Label Judi Online", "Pesan_Clean"])

In [12]:
train_base.head()

[Output tidak disertakan pada rilis publik. Hanya statistik agregat yang dipublikasikan; teks pesan pengguna, daftar frekuensi kata, dan identitas pengguna dihilangkan. Lihat 'Data & Content Policy' di README.]


In [13]:
import pandas as pd
import re
import unicodedata

# Cleaning
def clean_text(text):
    text = str(text)
    
    # Menghapus Emote bawaan YouTube (format :nama_emote:)
    text = re.sub(r':[^:\s]+:', ' ', text)

    # Normalisasi karakter Unicode (mengubah font unik/tebal/miring menjadi huruf standar)
    text = unicodedata.normalize('NFKD', text)
    
    # Menghapus karakter non-ASCII (Emoji standar, dll)
    text = text.encode('ascii', 'ignore').decode('ascii')
    
    # Menghapus URL/Link
    # text = re.sub(r'http\S+|www\.\S+', '', text)
    
    # Lowercase
    text = text.lower()
    
    # Menghapus angka dan tanda baca
    # text = re.sub(r'[^a-z\s]', ' ', text)
    
    # Menghapus ekstra spasi ganda dan spasi di awal/akhir kalimat
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

train_base['Pesan_Clean'] = train_base['Pesan'].apply(clean_text)
print(f"Jumlah pesan: {len(train_base)}")

# Simpan file
train_base.to_csv('../resources/train_base.csv', index=False)

Jumlah pesan: 55643


In [14]:
train_base.head()

[Output tidak disertakan pada rilis publik. Hanya statistik agregat yang dipublikasikan; teks pesan pengguna, daftar frekuensi kata, dan identitas pengguna dihilangkan. Lihat 'Data & Content Policy' di README.]


In [ ]:
import pandas as pd
import json
import time
from openai import OpenAI

# OpenRouter Config
OPENROUTER_API_KEY = "..."

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

MODEL_NAME = "google/gemini-2.5-flash-lite"

# Load data
df = pd.read_csv('../resources/train_base.csv')
print(f"Jumlah data: {len(df)}")

# Fungsi untuk memproses satu batch dengan OpenRouter
def label_batch_with_openrouter(batch_df):
    pesan_list = []
    for index, row in batch_df.iterrows():
        pesan_list.append({"id": index, "pesan": row['Pesan_Clean']})
    
    data_json_str = json.dumps(pesan_list, ensure_ascii=False)
    
    prompt = f"""
    Anda adalah asisten AI untuk mengklasifikasikan pesan live chat.
    Evaluasi setiap pesan dan berikan penilaian untuk 2 kategori terpisah:
    
    1. Kategori Toxic: Beri nilai "toxic" (jika ada kata kasar, makian, hinaan) atau "normal" (jika tidak ada).
    2. Kategori Judi Online: Beri nilai "judi online" (jika ada promosi judi, slot, depo, gacor, zeus, rtp, dll) atau "normal" (jika tidak berkaitan).

    Ini adalah daftar pesannya:
    {data_json_str}

    KEMBALIKAN OUTPUT HANYA DALAM FORMAT JSON ARRAY SEPERTI INI:
    [
      {{"id": 0, "label_toxic": "normal", "label_judi_online": "normal"}},
      {{"id": 1, "label_toxic": "toxic", "label_judi_online": "normal"}},
      {{"id": 2, "label_toxic": "normal", "label_judi_online": "judi online"}},
      {{"id": 3, "label_toxic": "toxic", "label_judi_online": "judi online"}}
    ]
    JANGAN tambahkan teks penjelasan apapun, murni JSON valid.
    """
    
    try:
        # Request OpenRouter
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": "Anda adalah mesin pengklasifikasi teks yang hanya membalas dengan format JSON array valid."},
                {"role": "user", "content": prompt}
            ],
            extra_headers={
                "HTTP-Referer": "https://localhost", 
                "X-Title": "PelabelanDataset", 
            }
        )

        raw_text = response.choices[0].message.content.strip()
        
        # Bersihkan format markdown
        if raw_text.startswith("```json"):
            raw_text = raw_text[7:]
        if raw_text.endswith("```"):
            raw_text = raw_text[:-3]
            
        # Parse ke JSON Python
        hasil_json = json.loads(raw_text.strip())
        return hasil_json
        
    except Exception as e:
        print(f"Error pada batch: {e}")
        if 'raw_text' in locals():
            print(f"Raw response: {raw_text}")
        return None

BATCH_SIZE = 30
hasil_akhir_toxic = {}
hasil_akhir_judi = {}

print(f"Mulai pelabelan dengan model {MODEL_NAME}...")
for i in range(0, len(df), BATCH_SIZE):
    batch_df = df.iloc[i : i + BATCH_SIZE]
    
    hasil_batch = label_batch_with_openrouter(batch_df)
    
    if hasil_batch:
        for item in hasil_batch:
            if 'id' in item and 'label_toxic' in item and 'label_judi_online' in item:
                hasil_akhir_toxic[item['id']] = item['label_toxic']
                hasil_akhir_judi[item['id']] = item['label_judi_online']
    
    print(f"Berhasil memproses baris {i} sampai {i + len(batch_df) - 1}")
    time.sleep(2)

# Gabungkan hasil menjadi DataFrame
df['Label Toxic'] = df.index.map(hasil_akhir_toxic)
df['Label Toxic'] = df['Label Toxic'].fillna('error')

df['Label Judi Online'] = df.index.map(hasil_akhir_judi)
df['Label Judi Online'] = df['Label Judi Online'].fillna('error')

# Save CSV
df.to_csv('../resources/train_gemini.csv', index=False)
print("\nSelesai! File disimpan sebagai 'train_gemini.csv'")

print("\n=== Distribusi Label Toxic ===")
print(df['Label Toxic'].value_counts())

print("\n=== Distribusi Label Judi Online ===")
print(df['Label Judi Online'].value_counts())

Jumlah data: 55643
Mulai pelabelan dengan model google/gemini-2.5-flash-lite...
Berhasil memproses baris 0 sampai 29
Berhasil memproses baris 30 sampai 59
Berhasil memproses baris 60 sampai 89
Berhasil memproses baris 90 sampai 119
Berhasil memproses baris 120 sampai 149
Berhasil memproses baris 150 sampai 179
Berhasil memproses baris 180 sampai 209
Berhasil memproses baris 210 sampai 239
Berhasil memproses baris 240 sampai 269
Berhasil memproses baris 270 sampai 299
Berhasil memproses baris 300 sampai 329
Berhasil memproses baris 330 sampai 359
Berhasil memproses baris 360 sampai 389
Berhasil memproses baris 390 sampai 419
Berhasil memproses baris 420 sampai 449
Berhasil memproses baris 450 sampai 479
Berhasil memproses baris 480 sampai 509
Berhasil memproses baris 510 sampai 539
Berhasil memproses baris 540 sampai 569
Berhasil memproses baris 570 sampai 599
Berhasil memproses baris 600 sampai 629
Berhasil memproses baris 630 sampai 659
Berhasil memproses baris 660 sampai 689
Berhasil

In [1]:
import pandas as pd

df = pd.read_csv("../resources/train_gemini.csv")

print("\n=== Distribusi Label Toxic ===")
print(df['Label Toxic'].value_counts())

print("\n=== Distribusi Label Judi Online ===")
print(df['Label Judi Online'].value_counts())


=== Distribusi Label Toxic ===
Label Toxic
normal    52743
toxic      2900
Name: count, dtype: int64

=== Distribusi Label Judi Online ===
Label Judi Online
normal         52641
judi online     3002
Name: count, dtype: int64


In [ ]:
# Deepseek V4 Flash

import pandas as pd
import json
import time
from openai import OpenAI

# OpenRouter Config
OPENROUTER_API_KEY = "..."

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

MODEL_NAME = "deepseek/deepseek-v4-flash"

# Load data
df = pd.read_csv('../resources/train_base.csv')
print(f"Jumlah data: {len(df)}")

# Fungsi untuk memproses satu batch dengan OpenRouter
def label_batch_with_openrouter(batch_df):
    pesan_list = []
    for index, row in batch_df.iterrows():
        pesan_list.append({"id": index, "pesan": row['Pesan_Clean']})
    
    data_json_str = json.dumps(pesan_list, ensure_ascii=False)
    
    prompt = f"""
    Anda adalah asisten AI untuk mengklasifikasikan pesan live chat.
    Evaluasi setiap pesan dan berikan penilaian untuk 2 kategori terpisah:
    
    1. Kategori Toxic: Beri nilai "toxic" (jika ada kata kasar, makian, hinaan) atau "normal" (jika tidak ada).
    2. Kategori Judi Online: Beri nilai "judi online" (jika ada promosi judi, slot, depo, gacor, zeus, rtp, dll) atau "normal" (jika tidak berkaitan).

    Ini adalah daftar pesannya:
    {data_json_str}

    KEMBALIKAN OUTPUT HANYA DALAM FORMAT JSON ARRAY SEPERTI INI:
    [
      {{"id": 0, "label_toxic": "normal", "label_judi_online": "normal"}},
      {{"id": 1, "label_toxic": "toxic", "label_judi_online": "normal"}},
      {{"id": 2, "label_toxic": "normal", "label_judi_online": "judi online"}},
      {{"id": 3, "label_toxic": "toxic", "label_judi_online": "judi online"}}
    ]
    JANGAN tambahkan teks penjelasan apapun, murni JSON valid.
    """
    
    try:
        # Request OpenRouter
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": "Anda adalah mesin pengklasifikasi teks yang hanya membalas dengan format JSON array valid."},
                {"role": "user", "content": prompt}
            ],
            extra_headers={
                "HTTP-Referer": "https://localhost", 
                "X-Title": "PelabelanDataset", 
            }
        )

        raw_text = response.choices[0].message.content.strip()
        
        # Bersihkan format markdown
        if raw_text.startswith("```json"):
            raw_text = raw_text[7:]
        if raw_text.endswith("```"):
            raw_text = raw_text[:-3]
            
        # Parse ke JSON Python
        hasil_json = json.loads(raw_text.strip())
        return hasil_json
        
    except Exception as e:
        print(f"Error pada batch: {e}")
        if 'raw_text' in locals():
            print(f"Raw response: {raw_text}")
        return None

BATCH_SIZE = 30
hasil_akhir_toxic = {}
hasil_akhir_judi = {}

print(f"Mulai pelabelan dengan model {MODEL_NAME}...")
for i in range(0, len(df), BATCH_SIZE):
    batch_df = df.iloc[i : i + BATCH_SIZE]
    
    hasil_batch = label_batch_with_openrouter(batch_df)
    
    if hasil_batch:
        for item in hasil_batch:
            if 'id' in item and 'label_toxic' in item and 'label_judi_online' in item:
                hasil_akhir_toxic[item['id']] = item['label_toxic']
                hasil_akhir_judi[item['id']] = item['label_judi_online']
    
    print(f"Berhasil memproses baris {i} sampai {i + len(batch_df) - 1}")
    time.sleep(2)

# Gabungkan hasil menjadi DataFrame
df['Label Toxic'] = df.index.map(hasil_akhir_toxic)
df['Label Toxic'] = df['Label Toxic'].fillna('error')

df['Label Judi Online'] = df.index.map(hasil_akhir_judi)
df['Label Judi Online'] = df['Label Judi Online'].fillna('error')

# Save CSV
df.to_csv('../resources/train_deepseek.csv', index=False)
print("\nSelesai! File disimpan sebagai 'train_deepseek.csv'")

print("\n=== Distribusi Label Toxic ===")
print(df['Label Toxic'].value_counts())

print("\n=== Distribusi Label Judi Online ===")
print(df['Label Judi Online'].value_counts())

Jumlah data: 55643
Mulai pelabelan dengan model deepseek/deepseek-v4-flash...
Berhasil memproses baris 44190 sampai 44219

Selesai! File disimpan sebagai 'train_deepseek_44190.csv'

=== Distribusi Label Toxic ===
Label Toxic
error     55613
normal       30
Name: count, dtype: int64

=== Distribusi Label Judi Online ===
Label Judi Online
error          55613
normal            25
judi online        5
Name: count, dtype: int64


In [7]:
import pandas as pd

df = pd.read_csv("../resources/train_deepseek.csv")

print("\n=== Distribusi Label Toxic ===")
print(df['Label Toxic'].value_counts())

print("\n=== Distribusi Label Judi Online ===")
print(df['Label Judi Online'].value_counts())


=== Distribusi Label Toxic ===
Label Toxic
normal    53061
toxic      2582
Name: count, dtype: int64

=== Distribusi Label Judi Online ===
Label Judi Online
normal         52518
judi online     3125
Name: count, dtype: int64


In [16]:
import pandas as pd

df_judi = pd.read_csv('../generate_label/output_from_llm/data_judi_online_komentar_youtube_cleaned_labeled.csv')
df_train = pd.read_csv('../resources/train_gemini.csv')

print(f"Jumlah data awal train_gemini: {len(df_train)} baris")

df_judi_subset = df_judi[['Waktu', 'Username', 'Pesan', 'Label Judi Online']]

merged_df = pd.merge(df_train, df_judi_subset, on=['Waktu', 'Username', 'Pesan'], how='left', suffixes=('', '_dari_judi'))
merged_df['Label Judi Online'] = merged_df['Label Judi Online_dari_judi'].combine_first(merged_df['Label Judi Online'])

merged_df = merged_df.drop(columns=['Label Judi Online_dari_judi'])

print(f"Jumlah data setelah digabung: {len(merged_df)} baris")

merged_df.to_csv('../resources/train_gemini_final.csv', index=False)
print("File berhasil disimpan sebagai '../resources/train_gemini_final.csv'")

Jumlah data awal train_gemini: 55643 baris
Jumlah data setelah digabung: 55643 baris
File berhasil disimpan sebagai '../resources/train_gemini_final.csv'


In [17]:
import pandas as pd

df = pd.read_csv("../resources/train_gemini_final.csv")

print("\n=== Distribusi Label Toxic ===")
print(df['Label Toxic'].value_counts())

print("\n=== Distribusi Label Judi Online ===")
print(df['Label Judi Online'].value_counts())


=== Distribusi Label Toxic ===
Label Toxic
normal    52743
toxic      2900
Name: count, dtype: int64

=== Distribusi Label Judi Online ===
Label Judi Online
normal         52170
judi online     3473
Name: count, dtype: int64


In [18]:
import pandas as pd

df_judi = pd.read_csv('../generate_label/output_from_llm/data_judi_online_youtube_wayang_cleaned_labeled.csv')
df_train = pd.read_csv('../resources/train_gemini_final.csv')

print(f"Jumlah data awal train_gemini: {len(df_train)} baris")

df_judi_subset = df_judi[['Waktu', 'Username', 'Pesan', 'Label Judi Online']]

merged_df = pd.merge(df_train, df_judi_subset, on=['Waktu', 'Username', 'Pesan'], how='left', suffixes=('', '_dari_judi'))
merged_df['Label Judi Online'] = merged_df['Label Judi Online_dari_judi'].combine_first(merged_df['Label Judi Online'])

merged_df = merged_df.drop(columns=['Label Judi Online_dari_judi'])

print(f"Jumlah data setelah digabung: {len(merged_df)} baris")

merged_df.to_csv('../resources/train_gemini_final.csv', index=False)
print("File berhasil disimpan sebagai '../resources/train_gemini_final.csv'")

Jumlah data awal train_gemini: 55643 baris
Jumlah data setelah digabung: 55643 baris
File berhasil disimpan sebagai '../resources/train_gemini_final.csv'


In [19]:
import pandas as pd

df = pd.read_csv("../resources/train_gemini_final.csv")

print("\n=== Distribusi Label Toxic ===")
print(df['Label Toxic'].value_counts())

print("\n=== Distribusi Label Judi Online ===")
print(df['Label Judi Online'].value_counts())


=== Distribusi Label Toxic ===
Label Toxic
normal    52743
toxic      2900
Name: count, dtype: int64

=== Distribusi Label Judi Online ===
Label Judi Online
normal         52344
judi online     3299
Name: count, dtype: int64


In [20]:
import pandas as pd

df_judi = pd.read_csv('../generate_label/output_from_llm/data_judi_online_komentar_youtube_cleaned_labeled.csv')
df_train = pd.read_csv('../resources/train_deepseek.csv')

print(f"Jumlah data awal train_deepseek: {len(df_train)} baris")

df_judi_subset = df_judi[['Waktu', 'Username', 'Pesan', 'Label Judi Online']]

merged_df = pd.merge(df_train, df_judi_subset, on=['Waktu', 'Username', 'Pesan'], how='left', suffixes=('', '_dari_judi'))
merged_df['Label Judi Online'] = merged_df['Label Judi Online_dari_judi'].combine_first(merged_df['Label Judi Online'])

merged_df = merged_df.drop(columns=['Label Judi Online_dari_judi'])

print(f"Jumlah data setelah digabung: {len(merged_df)} baris")

merged_df.to_csv('../resources/train_deepseek_final.csv', index=False)
print("File berhasil disimpan sebagai '../resources/train_deepseek_final.csv'")

Jumlah data awal train_deepseek: 55643 baris
Jumlah data setelah digabung: 55643 baris
File berhasil disimpan sebagai '../resources/train_deepseek_final.csv'


In [21]:
import pandas as pd

df = pd.read_csv("../resources/train_deepseek_final.csv")

print("\n=== Distribusi Label Toxic ===")
print(df['Label Toxic'].value_counts())

print("\n=== Distribusi Label Judi Online ===")
print(df['Label Judi Online'].value_counts())


=== Distribusi Label Toxic ===
Label Toxic
normal    53061
toxic      2582
Name: count, dtype: int64

=== Distribusi Label Judi Online ===
Label Judi Online
normal         52248
judi online     3395
Name: count, dtype: int64


In [22]:
import pandas as pd

df_judi = pd.read_csv('../generate_label/output_from_llm/data_judi_online_youtube_wayang_cleaned_labeled.csv')
df_train = pd.read_csv('../resources/train_deepseek_final.csv')

print(f"Jumlah data awal train_gemini: {len(df_train)} baris")

df_judi_subset = df_judi[['Waktu', 'Username', 'Pesan', 'Label Judi Online']]

merged_df = pd.merge(df_train, df_judi_subset, on=['Waktu', 'Username', 'Pesan'], how='left', suffixes=('', '_dari_judi'))
merged_df['Label Judi Online'] = merged_df['Label Judi Online_dari_judi'].combine_first(merged_df['Label Judi Online'])

merged_df = merged_df.drop(columns=['Label Judi Online_dari_judi'])

print(f"Jumlah data setelah digabung: {len(merged_df)} baris")

merged_df.to_csv('../resources/train_deepseek_final.csv', index=False)
print("File berhasil disimpan sebagai '../resources/train_deepseek_final.csv'")

Jumlah data awal train_gemini: 55643 baris
Jumlah data setelah digabung: 55643 baris
File berhasil disimpan sebagai '../resources/train_deepseek_final.csv'


In [ ]:
import pandas as pd

df = pd.read_csv("../resources/train_deepseek_final.csv")

print("\n=== Distribusi Label Toxic ===")
print(df['Label Toxic'].value_counts())

print("\n=== Distribusi Label Judi Online ===")
print(df['Label Judi Online'].value_counts())


=== Distribusi Label Toxic ===
Label Toxic
normal    53061
toxic      2582
Name: count, dtype: int64

=== Distribusi Label Judi Online ===
Label Judi Online
normal         52405
judi online     3238
Name: count, dtype: int64


In [24]:
import pandas as pd

df_ds = pd.read_csv('../resources/train_deepseek_final.csv')
df_gm = pd.read_csv('../resources/train_gemini_final.csv')
df_gpt = pd.read_csv('../resources/train_gpt.csv')

judi_df = pd.DataFrame({
    'ds': df_ds['Label Judi Online'],
    'gm': df_gm['Label Judi Online'],
    'gpt': df_gpt['Label Judi Online']
})

toxic_df = pd.DataFrame({
    'ds': df_ds['Label Toxic'],
    'gm': df_gm['Label Toxic'],
    'gpt': df_gpt['Label Toxic']
})

final_judi = judi_df.mode(axis=1)[0]
final_toxic = toxic_df.mode(axis=1)[0]

df_final = df_ds[['Waktu', 'Username', 'Pesan', 'data_source', 'Pesan_Clean']].copy()
df_final['Label Toxic'] = final_toxic
df_final['Label Judi Online'] = final_judi

df_final = df_final[['Waktu', 'Username', 'Pesan', 'data_source', 'Pesan_Clean', 'Label Toxic', 'Label Judi Online']]

output_filename = '../resources/train_final.csv'
df_final.to_csv(output_filename, index=False)

print(f"File berhasil disimpan sebagai {output_filename}")

File berhasil disimpan sebagai ../resources/train_final.csv


In [25]:
import pandas as pd

df = pd.read_csv("../resources/train_final.csv")

print("\n=== Distribusi Label Toxic ===")
print(df['Label Toxic'].value_counts())

print("\n=== Distribusi Label Judi Online ===")
print(df['Label Judi Online'].value_counts())


=== Distribusi Label Toxic ===
Label Toxic
normal    53394
toxic      2249
Name: count, dtype: int64

=== Distribusi Label Judi Online ===
Label Judi Online
normal         52456
judi online     3187
Name: count, dtype: int64


In [28]:
df.head(30)

[Output tidak disertakan pada rilis publik. Hanya statistik agregat yang dipublikasikan; teks pesan pengguna, daftar frekuensi kata, dan identitas pengguna dihilangkan. Lihat 'Data & Content Policy' di README.]
